# SQL Fundamentals with DuckDB

Your first SQL queries — no prior SQL experience needed.

Every query follows this pattern:

```sql
SELECT  what_you_want
FROM    table_name
WHERE   condition       -- filter rows   (optional)
GROUP BY column         -- group rows    (optional)
ORDER BY column DESC    -- sort result   (optional)
LIMIT   n               -- cap rows      (optional)
```

We query the **South Africa Census 2022** household sample using DuckDB, a fast SQL engine that
runs inside the notebook and reads pandas DataFrames directly.

The data uses **numeric codes**:
- `Province`: 1 Western Cape, 2 Eastern Cape, 3 Northern Cape, 4 Free State,
  5 KwaZulu-Natal, 6 North West, 7 Gauteng, 8 Mpumalanga, 9 Limpopo
- `Geo_type`: 1 Urban area, 2 Tribal/traditional area, 3 Farm area
- `DERH_HSIZE`: household size (10 means "10 or more"); `DERH_HHAGE`: age of the household head

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import duckdb

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))

from src.utilities.project_paths import RAW_DIR

CENSUS_DIR = RAW_DIR / 'south_africa' / 'Census2022SampleSTATA'

# The households table = Census2022Households joined to Census2022Geography on QID.
# Both files are modest (~60 MB + ~26 MB) -- we never touch the 347 MB Persons file here.
# convert_categoricals=False keeps every value as a numeric code (so Geo_type = 1, etc.)
hh = pd.read_stata(
    CENSUS_DIR / 'Census2022Households.dta',
    columns=['QID', 'DERH_HSIZE', 'DERH_HHAGE'],
    convert_categoricals=False,
)
geo = pd.read_stata(
    CENSUS_DIR / 'Census2022Geography.dta',
    columns=['QID', 'Province', 'District', 'Geo_type'],
    convert_categoricals=False,
)
households = (
    hh.merge(geo, on='QID', how='left')
      [['QID', 'Province', 'District', 'DERH_HSIZE', 'DERH_HHAGE', 'Geo_type']]
)

duckdb.register('households', households)

print(f'households : {len(households):,} rows')
print('columns    :', list(households.columns))

---
# Part A — Looking at the Data

| Clause | What it does |
|---|---|
| `SELECT *` | Return all columns |
| `SELECT col1, col2` | Return specific columns |
| `LIMIT n` | Return only the first n rows |

## A1. How many rows are there?

`COUNT(*)` counts every row in the table.

In [ ]:
# TODO: count all rows in the households table, name the result n_rows
duckdb.sql("""
    SELECT COUNT(*) AS ...
    FROM households
""").to_df()

## A2. Peek at the data

Show the first 5 rows with all columns.

In [ ]:
# TODO: select all columns, limit to 5 rows
duckdb.sql("""
    SELECT ...
    FROM households
    LIMIT ...
""").to_df()

## A3. Select specific columns

Show only `QID`, `Province`, and `DERH_HSIZE` for the first 10 rows.

In [ ]:
# TODO: select those three columns, limit to 10 rows
duckdb.sql("""
    SELECT ..., ..., ...
    FROM households
    LIMIT 10
""").to_df()

---
# Part B — Filtering with WHERE

`WHERE` keeps only rows that match a condition.

| Operator | Meaning | Example |
|---|---|---|
| `=` | equal | `Geo_type = 1` |
| `>` / `<` | greater / less than | `DERH_HSIZE > 5` |
| `>=` / `<=` | greater or equal / less or equal | `DERH_HSIZE >= 3` |
| `AND` | both conditions must be true | `Geo_type = 1 AND DERH_HSIZE > 5` |
| `OR` | at least one must be true | `Province = 7 OR Province = 8` |

## B1. Urban households only

`Geo_type = 1` means the household is in an urban area.
Show all columns, limit to 10 rows.

In [ ]:
# TODO: add a WHERE clause to filter Geo_type = 1
duckdb.sql("""
    SELECT *
    FROM households
    WHERE ...
    LIMIT 10
""").to_df()

## B2. Large households

Show households with more than 6 members.
Select `QID`, `Province`, `DERH_HSIZE`.

In [ ]:
# TODO: WHERE DERH_HSIZE > 6
duckdb.sql("""
    SELECT QID, Province, DERH_HSIZE
    FROM households
    WHERE ...
""").to_df()

## B3. Combine two conditions

Show large households (> 6 members) in urban areas only.
Use `AND` to apply both filters at once.

In [ ]:
# TODO: WHERE Geo_type = 1 AND DERH_HSIZE > 6
# SELECT QID, Province, DERH_HSIZE, DERH_HHAGE
duckdb.sql("""
    SELECT ...
    FROM households
    WHERE ... AND ...
""").to_df()

---
# Part C — Counting and Summarising

Aggregate functions collapse many rows into a single number.

| Function | Result |
|---|---|
| `COUNT(*)` | number of rows |
| `AVG(col)` | mean |
| `SUM(col)` | total |
| `MIN(col)` | smallest value |
| `MAX(col)` | largest value |

You can use several aggregates in a single `SELECT`.

## C1. How many urban households?

Count only the rows where `Geo_type = 1`.

In [ ]:
# TODO: WHERE Geo_type = 1, COUNT(*) AS n_urban
duckdb.sql("""
    SELECT COUNT(*) AS n_urban
    FROM households
    WHERE ...
""").to_df()

## C2. Summarise household size

Compute the average, minimum, and maximum of `DERH_HSIZE` across all rows.

In [ ]:
# TODO: SELECT AVG(...) AS mean_size, MIN(...) AS min_size, MAX(...) AS max_size
duckdb.sql("""
    SELECT
        AVG(DERH_HSIZE) AS mean_size,
        ...
    FROM households
""").to_df()

## C3. Multiple aggregations on filtered data

For urban households: count households, mean household size, and total number of people
(the sum of household sizes).

In [ ]:
# TODO: WHERE Geo_type = 1
# SELECT COUNT(*) AS n_hh, AVG(DERH_HSIZE) AS mean_size, SUM(DERH_HSIZE) AS total_people
duckdb.sql("""
    SELECT
        COUNT(*) AS n_hh,
        ...
    FROM households
    WHERE ...
""").to_df()

---
# Part D — Grouping with GROUP BY

`GROUP BY` splits rows into groups and applies aggregate functions to each group.

```sql
SELECT  group_column, AGG(value_column) AS alias
FROM    table
WHERE   condition
GROUP BY group_column
```

**Rule:** every column in `SELECT` must be either in `GROUP BY` or wrapped in an aggregate.

## D1. Count households per geo type

How many households have each value of `Geo_type` (1 urban, 2 tribal/traditional, 3 farm)?

In [ ]:
# TODO: GROUP BY Geo_type, COUNT(*) AS n_hh
duckdb.sql("""
    SELECT
        Geo_type,
        COUNT(*) AS n_hh
    FROM households
    GROUP BY ...
""").to_df()

## D2. Average household size by province

For urban households, compute the number of households and mean household size in each province.

In [ ]:
# TODO:
# SELECT Province, COUNT(*) AS n_hh, AVG(DERH_HSIZE) AS mean_size
# WHERE Geo_type = 1
# GROUP BY Province
duckdb.sql("""
    SELECT
        Province,
        ...
    FROM households
    WHERE ...
    GROUP BY ...
""").to_df()

---
# Part E — Sorting with ORDER BY

`ORDER BY` sorts the result rows.

```sql
ORDER BY column_name        -- ascending (smallest first, this is the default)
ORDER BY column_name DESC   -- descending (largest first)
```

Combine with `LIMIT` to get the top or bottom N rows.

## E1. Provinces with the largest average household size

Repeat D2, but sort by `mean_size` descending so the largest provinces appear first.

In [ ]:
# TODO: add ORDER BY mean_size DESC to the D2 query
duckdb.sql("""
    SELECT
        Province,
        COUNT(*)        AS n_hh,
        AVG(DERH_HSIZE) AS mean_size
    FROM households
    WHERE Geo_type = 1
    GROUP BY Province
    ORDER BY ...
""").to_df()

## E2. The 5 oldest household heads

Among urban households, find the 5 whose head is oldest.
Show `QID`, `Province`, `DERH_HHAGE`.

In [ ]:
# TODO: ORDER BY DERH_HHAGE DESC, LIMIT 5
duckdb.sql("""
    SELECT QID, Province, DERH_HHAGE
    FROM households
    WHERE Geo_type = 1
    ORDER BY ...
    LIMIT ...
""").to_df()

## E3. Provinces with the fewest urban households

Sort ascending (`ASC`, the default) to find the smallest provinces first.

In [ ]:
# TODO: GROUP BY Province, COUNT(*) AS n_hh, ORDER BY n_hh ASC
duckdb.sql("""
    SELECT
        Province,
        COUNT(*) AS n_hh
    FROM households
    WHERE Geo_type = 1
    GROUP BY Province
    ORDER BY ...
""").to_df()